# ENSEMBLE LEARNING

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob, os
import copy

from sklearn.metrics import roc_auc_score, f1_score
#Models import
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier 
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

#Preprocessing
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Binarizer

from mlxtend.feature_selection import SequentialFeatureSelector as SFS

# 1. Xử lí dữ liệu

In [2]:
folder_path = "./data/dataNasa"  # đường dẫn tới thư mục chứa các file CSV
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
csv_files
data_dict = {}

for file in csv_files:
    filename = os.path.basename(file)
    df = pd.read_csv(file)
    data_dict[filename] = df
file_names = list(data_dict.keys())
data_dict.keys()


dict_keys(['CM1.csv', 'KC1.csv', 'KC3.csv', 'MC1.csv', 'MC2.csv', 'MW1.csv', 'PC1.csv', 'PC3.csv', 'PC4.csv', 'PC5.csv'])

In [3]:
for data in data_dict.values():
    data.drop(columns=['id'], inplace=True)
    data["Defective"] = data["Defective"].map({"Y": 1, "N": 0})


In [4]:
# Chuẩn hóa Gaussian 
scaler1 = StandardScaler()
data_dict1 = copy.deepcopy(data_dict)
for key in data_dict1.keys():
    features = data_dict1[key].columns[:-1]
    data_dict1[key][features] = scaler1.fit_transform(data_dict1[key][features])

data_dict1['CM1.csv'].describe()

,LOC_BLANK,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DECISION_DENSITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
count,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,327.000000,3.270000e+02,3.270000e+02,...,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,3.270000e+02,327.000000
mean,1.358071e-17,5.024863e-17,-4.074213e-18,2.172914e-17,2.172914e-17,-2.172914e-17,6.790355e-18,0.000000,3.259370e-17,1.303748e-16,...,-6.247126e-17,-2.390205e-16,7.061969e-17,8.148426e-18,2.105010e-17,9.778111e-17,5.568091e-17,-2.118591e-16,-3.123563e-17,0.128440
std,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533,1.001533e+00,1.001533e+00,...,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,1.001533e+00,0.335092
min,-7.250309e-01,-5.955341e-01,-9.642652e-01,-5.612562e-01,-5.735276e-01,-6.211983e-01,-5.639274e-01,-2.055046,-6.205568e-01,-3.722372e-01,...,-7.027537e-01,-1.523535e+00,-7.416405e-01,-7.508650e-01,-8.511192e-01,-1.624157e+00,-7.672575e-01,-1.575625e+00,-7.105966e-01,0.000000
25%,-5.520707e-01,-4.766091e-01,-7.101841e-01,-5.612562e-01,-5.407001e-01,-4.743206e-01,-4.580088e-01,-0.592056,-4.628862e-01,-3.722372e-01,...,-5.654979e-01,-6.249130e-01,-5.586783e-01,-5.452205e-01,-5.887797e-01,-6.553906e-01,-5.687247e-01,-8.358608e-01,-5.144176e-01,0.000000
50%,-3.358706e-01,-3.576842e-01,-2.020217e-01,-3.628446e-01,-3.109076e-01,-3.274429e-01,-3.520902e-01,-0.104393,-3.052156e-01,-3.722372e-01,...,-3.596143e-01,-2.654644e-01,-3.757160e-01,-3.462098e-01,-3.526741e-01,-2.248276e-01,-3.179464e-01,8.871081e-02,-3.360731e-01,0.000000
75%,1.397698e-01,1.180158e-01,3.061406e-01,3.397876e-02,9.943621e-02,1.131901e-01,7.158412e-02,0.383271,1.677962e-01,-1.066061e-01,...,1.894088e-01,4.534329e-01,1.579238e-01,1.081982e-01,1.720049e-01,4.210169e-01,8.434368e-02,8.271432e-01,1.097882e-01,0.000000
max,6.366334e+00,8.859002e+00,5.641845e+00,7.375210e+00,1.055500e+01,8.338340e+00,9.392420e+00,6.397785,8.524338e+00,9.002981e+00,...,7.738476e+00,7.103233e+00,7.329027e+00,7.355507e+00,7.307639e+00,5.695413e+00,7.121809e+00,2.255276e+00,8.135292e+00,1.000000


In [6]:
#Chuẩn hóa Min-Max
scaler2 = MinMaxScaler()
data_dict2 = copy.deepcopy(data_dict)
for key in data_dict2.keys():
    features = data_dict2[key].columns[:-1]
    data_dict2[key][features] = scaler2.fit_transform(data_dict2[key][features])

data_dict2['CM1.csv'].describe()

,LOC_BLANK,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DECISION_DENSITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
count,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,...,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000
mean,0.102241,0.062989,0.145966,0.070719,0.051537,0.069334,0.056640,0.243119,0.067858,0.039704,...,0.083253,0.176606,0.091893,0.092627,0.104320,0.221892,0.097256,0.411294,0.080331,0.128440
std,0.141233,0.105931,0.151607,0.126194,0.089997,0.111784,0.100592,0.118485,0.109518,0.106828,...,0.118648,0.116096,0.124095,0.123549,0.122756,0.136829,0.126952,0.261435,0.113220,0.335092
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.024390,0.012579,0.038462,0.000000,0.002950,0.016393,0.010638,0.173077,0.017241,0.000000,...,0.016260,0.104167,0.022670,0.025368,0.032154,0.132353,0.025166,0.193104,0.022177,0.000000
50%,0.054878,0.025157,0.115385,0.025000,0.023599,0.032787,0.021277,0.230769,0.034483,0.000000,...,0.040650,0.145833,0.045340,0.049918,0.061093,0.191176,0.056954,0.434450,0.042339,0.000000
75%,0.121951,0.075472,0.192308,0.075000,0.060472,0.081967,0.063830,0.288462,0.086207,0.028333,...,0.105691,0.229167,0.111461,0.105974,0.125402,0.279412,0.107947,0.627207,0.092742,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# 2. Greedy Feature Selection

In [7]:
def GFS(model, scoring, X, y):
    sfs = SFS(model, k_features='best', forward=True, floating=False, scoring = scoring, cv=5)
    sfs = sfs.fit(X, y)
    selected = list(sfs.k_feature_names_)
    return selected

# 3. 7 Classifier

Random Forest

In [12]:
RF_auc = []
k = 5  #5 fold

for i in range(len(file_names)):
    #Setup data
    X = data_dict[file_names[i]].drop(columns=['Defective'], axis = 1)
    y = data_dict[file_names[i]]['Defective']
    auc = 0
    rf = RandomForestClassifier(random_state=42)

    #Train với K-Fold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #GFS
        selected = GFS(rf, 'roc_auc', X_train, y_train)
        X_train = X_train[selected]
        X_test = X_test[selected]
        
        #Train model
        rf.fit(X_train, y_train)

        y_pred_prob = rf.predict_proba(X_test)[:, 1]
        auc = auc + roc_auc_score(y_test, y_pred_prob)

    RF_auc.append(auc/k)

In [15]:
RF_auc

[0.6722953216374269,
 0.7349303727156228,
 0.7210829493087556,
 0.8069579137137884,
 0.7142361111111111,
 0.7573333333333334,
 0.8304134897360704,
 0.8103750112989244,
 0.9351863737828925,
 0.8089164841155062]

Gradient Boosting

In [11]:
GB_auc = []
k = 5  #5 fold

for i in range(len(file_names)):
    #Setup data
    X = data_dict1[file_names[i]].drop(columns=['Defective'], axis = 1)
    y = data_dict1[file_names[i]]['Defective']
    auc = 0
    gb = GradientBoostingClassifier(random_state=42)
    
    #Train với K-Fold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        #GFS
        selected = GFS(gb, 'roc_auc', X_train, y_train)
        X_train = X_train[selected]
        X_test = X_test[selected]
        
        gb.fit(X_train, y_train)

        y_pred_prob = gb.predict_proba(X_test)[:, 1]
        auc = auc + roc_auc_score(y_test, y_pred_prob)
        
    GB_auc.append(auc/k)

In [14]:
GB_auc

[0.6402046783625731,
 0.6714619359053561,
 0.7161290322580645,
 0.856301481101579,
 0.7260416666666666,
 0.6768888888888889,
 0.8170357771260998,
 0.7953700397722135,
 0.9371547763674194,
 0.7805205068282048]

Stochastic Gradient Descent

In [10]:
SGD_auc = []
k = 5  #5 fold

for i in range(len(file_names)):
    #Setup data
    X = data_dict1[file_names[i]].drop(columns=['Defective'], axis = 1)
    y = data_dict1[file_names[i]]['Defective']
    auc = 0
    SGD = SGDClassifier(loss='log_loss', penalty='l2',
                        class_weight='balanced', max_iter=1000, random_state=42)
    
    #Train với K-Fold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #GFS
        selected = GFS(SGD, 'roc_auc', X_train, y_train)
        X_train = X_train[selected]
        X_test = X_test[selected]

        SGD.fit(X_train, y_train)

        y_pred_prob = SGD.predict_proba(X_test)[:, 1]
        auc = auc + roc_auc_score(y_test, y_pred_prob)
        
    SGD_auc.append(auc/k)

In [13]:
SGD_auc

[0.7206140350877193,
 0.6440529581072513,
 0.6876152073732718,
 0.7449546577769488,
 0.6670138888888889,
 0.7395555555555555,
 0.8236568914956012,
 0.7952266564223086,
 0.8639221829236912,
 0.7011964061002736]

Logistic Regression

In [8]:
LogR_auc = []
k = 5  #5 fold

for i in range(len(file_names)):
    #Setup data
    X = data_dict1[file_names[i]].drop(columns=['Defective'], axis = 1)
    y = data_dict1[file_names[i]]['Defective']
    auc = 0
    LogR = LogisticRegression(solver='liblinear', penalty='l2', 
                              class_weight='balanced', max_iter=1000, random_state=42)
    
    #Train với K-Fold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #GFS
        selected = GFS(LogR, 'roc_auc', X_train, y_train)
        X_train = X_train[selected]
        X_test = X_test[selected]

        LogR.fit(X_train, y_train)

        y_pred_prob = LogR.predict_proba(X_test)[:, 1]
        auc = auc + roc_auc_score(y_test, y_pred_prob)
        
    LogR_auc.append(auc/k)

In [13]:
LogR_auc

[0.6911793372319688,
 0.7118231288507056,
 0.7028225806451612,
 0.7847207975879646,
 0.7237847222222222,
 0.7386666666666667,
 0.8864633431085043,
 0.8304313929313929,
 0.9058686471530507,
 0.7572472259472219]

Weighted-SVM

In [9]:
wSVM_auc = []
k = 5  #5 fold

for i in range(len(file_names)):
    #Setup data
    X = data_dict1[file_names[i]].drop(columns=['Defective'], axis = 1)
    y = data_dict1[file_names[i]]['Defective']
    auc = 0
    wSVM = SVC(kernel = 'rbf', class_weight='balanced', probability = True, random_state=42)
    
    #Train với K-Fold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #GFS
        selected = GFS(wSVM, 'roc_auc', X_train, y_train)
        X_train = X_train[selected]
        X_test = X_test[selected]

        wSVM.fit(X_train, y_train)

        y_pred_prob = wSVM.predict_proba(X_test)[:, 1]
        auc = auc + roc_auc_score(y_test, y_pred_prob)
        
    wSVM_auc.append(auc/k)

In [15]:
wSVM_auc

[0.6423489278752437,
 0.6964447801301796,
 0.7425403225806452,
 0.726489649384558,
 0.7348958333333334,
 0.7368888888888889,
 0.8518005865102639,
 0.8251663201663202,
 0.8966504385676183,
 0.759755297779541]

Multinominal Naïve Bayes

In [11]:
MultiNB_auc = []
k = 5  #5 fold

for i in range(len(file_names)):
    #Setup data
    X = data_dict2[file_names[i]].drop(columns=['Defective'], axis = 1)
    y = data_dict2[file_names[i]]['Defective']
    auc = 0
    MultiNB = MultinomialNB(alpha = 0.1)
    
    #Train với K-Fold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #GFS
        selected = GFS(MultiNB, 'roc_auc', X_train, y_train)
        X_train = X_train[selected]
        X_test = X_test[selected]

        MultiNB.fit(X_train, y_train)

        y_pred_prob = MultiNB.predict_proba(X_test)[:, 1]
        auc = auc + roc_auc_score(y_test, y_pred_prob)
        
    MultiNB_auc.append(auc/k)

In [12]:
MultiNB_auc

[0.6755847953216374,
 0.6951713569926927,
 0.6891129032258064,
 0.7825537346139501,
 0.671701388888889,
 0.8222222222222222,
 0.8484422287390029,
 0.8122021603543341,
 0.8597218282394856,
 0.7457839378711016]

Bernoulli Naïve Bayes

In [8]:
#Xử lý dữ liệu cho BerNB
binarizer = Binarizer(threshold=0.0)
Binary_data_dict = copy.deepcopy(data_dict1)

for key in Binary_data_dict.keys():
    features = Binary_data_dict[key].columns[:-1]
    Binary_data_dict[key][features] = binarizer.fit_transform(Binary_data_dict[key][features])

Binary_data_dict['CM1.csv'].describe()
#data_dict1['CM1.csv'].describe()


,LOC_BLANK,BRANCH_COUNT,CALL_PAIRS,LOC_CODE_AND_COMMENT,LOC_COMMENTS,CONDITION_COUNT,CYCLOMATIC_COMPLEXITY,CYCLOMATIC_DENSITY,DECISION_COUNT,DECISION_DENSITY,...,NODE_COUNT,NORMALIZED_CYLOMATIC_COMPLEXITY,NUM_OPERANDS,NUM_OPERATORS,NUM_UNIQUE_OPERANDS,NUM_UNIQUE_OPERATORS,NUMBER_OF_LINES,PERCENT_COMMENTS,LOC_TOTAL,Defective
count,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,...,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000,327.000000
mean,0.302752,0.259939,0.418960,0.278287,0.308869,0.269113,0.262997,0.467890,0.314985,0.229358,...,0.305810,0.415902,0.293578,0.275229,0.318043,0.363914,0.284404,0.529052,0.278287,0.128440
std,0.460153,0.439273,0.494145,0.448843,0.462735,0.444179,0.440936,0.499733,0.465222,0.421064,...,0.461456,0.493632,0.456099,0.447314,0.466430,0.481862,0.451821,0.499920,0.448843,0.335092
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
BerNB_auc = []
k = 5  #5 fold

for i in range(len(file_names)):
    #Setup data
    X = Binary_data_dict[file_names[i]].drop(columns=['Defective'], axis = 1)
    y = Binary_data_dict[file_names[i]]['Defective']
    auc = 0
    BerNB = BernoulliNB(alpha = 0.1)

    #Train với K-Fold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #GFS
        selected = GFS(BerNB, 'roc_auc', X_train, y_train)
        X_train = X_train[selected]
        X_test = X_test[selected]        

        BerNB.fit(X_train, y_train)

        y_pred_prob = BerNB.predict_proba(X_test)[:, 1]
        auc = auc + roc_auc_score(y_test, y_pred_prob)

    BerNB_auc.append(auc/k)

In [10]:
BerNB_auc

[0.7499269005847953,
 0.6693002441610855,
 0.7386232718894009,
 0.7776155313626757,
 0.6733506944444445,
 0.7622222222222221,
 0.8168439882697947,
 0.8170273659947572,
 0.8382522875934338,
 0.7413630910600151]